# Text-to-SQL with LLMs (Chinook / SQLite)

This exercise folder layout matches the other labs: **`Excersices/Text_to_SQL_with_LLMs/`** with **`data/Chinook.db`**, this notebook, **`requirements-text-to-sql-exercise.txt`**, and **`Text_to_SQL_with_LLMs.docx`**.

**Pipeline:** open SQLite → **`SQLDatabase`** (inspect schema) → **`SQLDatabaseChain`** (LangChain’s *all-in-one* text-to-SQL + execute + answer) → **`ChatOllama`** *if you need full control over prompts* → optional **plain-Python** *retry* after failed SQL → **optional** **few-shot** examples (n=2, 3, or 4) in the prompt to steer small models.

**Prerequisites:** Ollama running, e.g. `ollama pull llama3.2:1b`. Optional: `OPENAI_API_KEY` in a `.env` at the course repo root.


In [15]:
%pip install -qU "sqlalchemy>=2" langchain-core langchain-community langchain-ollama langchain-experimental python-dotenv



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Part 0 — Paths, `Chinook.db`, optional `.env`


In [16]:
import sqlite3
import urllib.request
from pathlib import Path

_here = Path.cwd()
if _here.name == "Text_to_SQL_with_LLMs":
    CHINOOK_DB = _here / "data" / "Chinook.db"
elif _here.name == "Excersices":
    CHINOOK_DB = _here / "Text_to_SQL_with_LLMs" / "data" / "Chinook.db"
else:
    # cwd = course repository root (e.g. IE-2-Course)
    CHINOOK_DB = _here / "Excersices" / "Text_to_SQL_with_LLMs" / "data" / "Chinook.db"


def ensure_chinook_db(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 1000:
        print("Using existing", path)
        return
    url = (
        "https://raw.githubusercontent.com/lerocha/chinook-database/master/"
        "ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    )
    print("Downloading Chinook SQL script…")
    sql = urllib.request.urlopen(url, timeout=120).read().decode("utf-8")
    con = sqlite3.connect(path)
    con.executescript(sql)
    con.close()
    print("Wrote", path)


ensure_chinook_db(CHINOOK_DB)

# Optional API keys (course repo or exercise folder)
LAB_ROOT = CHINOOK_DB.parent.parent
try:
    from dotenv import load_dotenv
    for env in (LAB_ROOT / ".env", LAB_ROOT.parent.parent / ".env", _here / ".env"):
        if env.is_file():
            load_dotenv(env)
            print("Loaded .env from", env)
            break
except ImportError:
    pass

DB_URI = f"sqlite:///{CHINOOK_DB.resolve()}"
print("DB_URI =", DB_URI)


Using existing /Users/farhadzad/Project/IE-2-Course/Excersices/Text_to_SQL_with_LLMs/data/Chinook.db
Loaded .env from /Users/farhadzad/Project/IE-2-Course/.env
DB_URI = sqlite:////Users/farhadzad/Project/IE-2-Course/Excersices/Text_to_SQL_with_LLMs/data/Chinook.db


## Part 1 — `SQLDatabase`: names, **schema string**, and smoke runs

- **`get_usable_table_names()`** — which tables the chain is allowed to see.  
- **`get_table_info()`** — a long text block (column names, types) you can *truncate* and pass to a chat model as “schema context”.  
- **`db.run("SELECT ...")`** — run read-only SQL and get a string back (handy to print or re-feed to the LLM).


In [17]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri(DB_URI)
names = db.get_usable_table_names()
print("Tables (first 15 of", len(names), "):", names[:15])

# Schema blob used later in custom prompts; keep only a prefix for small models / short context
print("\n--- Schema (first 1800 chars) ---")
print(db.get_table_info()[:1800], "\n...")

# Smoke: named columns + a simple aggregate
print(db.run("SELECT Name FROM Artist LIMIT 5;"))
print(db.run("SELECT COUNT(*) AS track_count FROM Track;"))
print(db.run("SELECT a.Title, COUNT(t.TrackId) AS n_tracks FROM Album a JOIN Track t ON t.AlbumId = a.AlbumId GROUP BY a.AlbumId LIMIT 3;"))


Tables (first 15 of 11 ): ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']

--- Schema (first 1800 chars) ---

CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVAR

## Part 2 — `SQLDatabaseChain` + **`OllamaLLM`**

### What is `SQLDatabaseChain`?

It is a **LangChain (experimental) chain** class: **`SQLDatabaseChain.from_llm(llm, db, …)`** wires your **text** language model to a **`SQLDatabase`** object. You do **not** write the full prompt yourself; the chain uses a **built-in template** that injects **table/column information** from the database and asks the model to produce **SQL for your question**. The chain then **runs** that SQL via `db.run`, reads the result string, and (by default) asks the model again to produce a **short natural-language answer** from the result. So in one `invoke({"query": "…"})` you get: *question → (LLM) SQL → execute → (LLM) paraphrased answer*.

### What to watch for

- It expects a **completion** model: **`OllamaLLM`**, not **`ChatOllama`** (different message API).  
- You can **log** behavior with **`verbose=True`**, and inspect decisions with **`return_intermediate_steps=True`**.  
- **`top_k`**: cap on how many rows from the query result are passed into the *answer* step (not “how many examples”).  
- **`use_query_checker=True`**: an **extra** LLM pass that tries to **fix** the generated SQL before execution (slower, sometimes helpful on weak models).  
- It is a **convenient baseline**; for custom guardrails, tools, or chat-style systems you often move to Parts 3–4.

**Next cells** run the chain, then (optionally) a harder question with the query checker.


In [18]:
from langchain_ollama import OllamaLLM
from langchain_experimental.sql import SQLDatabaseChain

llm = OllamaLLM(model="llama3.2:1b", temperature=0)

# 2a — one-shot answer (most natural starting point)
db_chain = SQLDatabaseChain.from_llm(
    llm, db, verbose=True, return_intermediate_steps=True, top_k=8
)
q = "How many artists are in the database?"
result = db_chain.invoke({"query": q})
print("Final answer:\n", result.get("result", result))
# Inspecting steps helps when SQL is wrong or the answer misreads the table
if result.get("intermediate_steps"):
    for i, st in enumerate(result["intermediate_steps"]):
        print(f"\n[intermediate_steps[{i}]] {st!r}")




> Entering new SQLDatabaseChain chain...
How many artists are in the database?
SQLQuery:SELECT COUNT(T1.ArtistId) FROM Artist AS T1 INNER JOIN Album AS T2 ON T1.ArtistId = T2.ArtistId
SQLResult: [(347,)]
Answer:To find the number of artists in the database, we can use a query like this:

```sql
SELECT COUNT(T1.ArtistId) FROM Artist AS T1 INNER JOIN Album AS T2 ON T1.ArtistId = T2.ArtistId;
```

This query joins the `Artist` table with the `Album` table on the `ArtistId` column, and then counts the number of rows in the resulting table. The result will be a single value representing the total number of artists.

The answer to this question is: 347
> Finished chain.
Final answer:
 To find the number of artists in the database, we can use a query like this:

```sql
SELECT COUNT(T1.ArtistId) FROM Artist AS T1 INNER JOIN Album AS T2 ON T1.ArtistId = T2.ArtistId;
```

This query joins the `Artist` table with the `Album` table on the `ArtistId` column, and then counts the number of rows in 

In [ ]:
# 2b — harder join + query checker (optional; comment out if too slow on CPU) # more probably you will see error with small LLM, find where is the problem?

db_chain_check = SQLDatabaseChain.from_llm(
    llm,
    db,
    verbose=True,
    return_intermediate_steps=True,
    use_query_checker=True,  # extra LLM self-check on the SQL
    top_k=10,
)
q2 = "What are the three longest tracks? Return the track name and how many minutes long each is (use Milliseconds from Track)."
out2 = db_chain_check.invoke({"query": q2})
print("Answer:\n", out2.get("result", out2))
# Uncomment to see full step structure
# for i, st in enumerate(out2.get("intermediate_steps", [])):
#     print(f"--- step {i} ---", st)




> Entering new SQLDatabaseChain chain...
What are the three longest tracks? Return the track name and how many minutes long each is (use Milliseconds from Track).
SQLQuery:SELECT T1.Name, T2.Milliseconds FROM Track AS T1 INNER JOIN PlaylistTrack AS T2 ON T1.TrackId = T2.TrackId ORDER BY T2.Milliseconds DESC LIMIT 3
```sql
-- No common mistakes found.
```

However, here's the corrected query:

```sql
SELECT T1.Name, T2.Milliseconds 
FROM Track T1 
INNER JOIN PlaylistTrack T2 
ON T1.TrackId = T2.TrackId 
ORDER BY T2.Milliseconds DESC 
LIMIT 3;
```
Explanation of corrections:
- Using NOT IN with NULL values is not necessary and can be removed.
- UNION ALL should have been used instead of UNION, as they are mutually exclusive queries that cannot be executed at the same time. The corrected query uses UNION to combine the results of two separate queries.
- BETWEEN for exclusive ranges is incorrect. Instead, use BETWEEN to specify an inclusive range (e.g., 1 AND 100).
- Data type mismatch i

OperationalError: (sqlite3.OperationalError) near "```sql
-- No common mistakes found.
```": syntax error
[SQL: SELECT T1.Name, T2.Milliseconds FROM Track AS T1 INNER JOIN PlaylistTrack AS T2 ON T1.TrackId = T2.TrackId ORDER BY T2.Milliseconds DESC LIMIT 3
```sql
-- No common mistakes found.
```

However, here's the corrected query:

```sql
SELECT T1.Name, T2.Milliseconds 
FROM Track T1 
INNER JOIN PlaylistTrack T2 
ON T1.TrackId = T2.TrackId 
ORDER BY T2.Milliseconds DESC 
LIMIT 3;
```
Explanation of corrections:
- Using NOT IN with NULL values is not necessary and can be removed.
- UNION ALL should have been used instead of UNION, as they are mutually exclusive queries that cannot be executed at the same time. The corrected query uses UNION to combine the results of two separate queries.
- BETWEEN for exclusive ranges is incorrect. Instead, use BETWEEN to specify an inclusive range (e.g., 1 AND 100).
- Data type mismatch in predicates: The correct data types are INTEGER and REAL, res

## Part 3 — `ChatOllama` + **manual** `db.run` (you control the SQL)

### Why Part 3 (after the chain in Part 2)?

`SQLDatabaseChain` hides the **exact prompt** and the **order of steps**. In many real systems you need to:

- **Fix the system instruction** (dialect, “SELECT only”, no destructive statements, which tables to prefer).  
- **Format the schema** your way (truncate, table grouping, or retrieved snippets).  
- **Parse the model output** and run SQL **only after** your own checks, or hand it to a **tool** / API with logging.

**Part 3** is the minimal **“I own the prompt + I execute with `db.run`”** pattern, close to how **tool-calling** or **RAG over schema** would plug in. **Part 4** then adds a **second turn** when execution fails (retry with the error text)—again under your control.

You build a **system** message; the model’s reply is a string; you **extract** a `SELECT` with a regex, then **`db.run(sql)`** yourself.

**After Part 3:** an **optional** cell adds **few-shot** (n=2, 3, or 4) *in-context* examples. **Part 4** packages generate → run → repair in one function.


In [20]:
import re
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

chat = ChatOllama(model="llama3.2:1b", temperature=0)
schema_snippet = db.get_table_info()[:4000]  # truncate for small context windows

user_q = "List 5 track names together with the album title (JOIN Album)."
messages = [
    SystemMessage(
        content="You output ONLY one SQLite SELECT query, no explanation. Dialect: SQLite. No semicolon required."
    ),
    HumanMessage(
        content=f"Schema (truncated):\n{schema_snippet}\n\nQuestion: {user_q}"
    ),
]
raw = chat.invoke(messages).content
print("Model output:\n", raw)
m = re.search(r"(SELECT[\s\S]+?)(;|$)", raw, re.I)
if not m:
    print("No SELECT found; try a larger model or shorter schema.")
else:
    sql = m.group(1).strip()
    print("--- Executing ---\n", sql)
    print(db.run(sql))


Model output:
 SELECT T1.Title, T3.Name FROM Album AS T1 INNER JOIN InvoiceLine AS T2 ON T1.AlbumId = T2.InvoiceId INNER JOIN Genre AS T3 ON T2.GenreId = T3.GenreId
--- Executing ---
 SELECT T1.Title, T3.Name FROM Album AS T1 INNER JOIN InvoiceLine AS T2 ON T1.AlbumId = T2.InvoiceId INNER JOIN Genre AS T3 ON T2.GenreId = T3.GenreId


OperationalError: (sqlite3.OperationalError) no such column: T2.GenreId
[SQL: SELECT T1.Title, T3.Name FROM Album AS T1 INNER JOIN InvoiceLine AS T2 ON T1.AlbumId = T2.InvoiceId INNER JOIN Genre AS T3 ON T2.GenreId = T3.GenreId]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

### (Optional) Few-shot in-context learning — set **n = 2, 3, or 4**

**Idea:** put **`n` short examples** in the *same* prompt: each example is a **user question** and a **correct SQLite `SELECT`** for this database (or a similar style). The model’s task is to mimic that pattern on a **new** question. This is **in-context learning** (not weight updates): more tokens, but often better SQL from **small** local models. Try `N = 2` first, then `3` or `4` if the context window allows; compare accuracy vs. Part 3 without examples.

**Note:** `SQLDatabaseChain` can be customized with a *custom* `PromptTemplate` in code, but we keep few-shot in the **manual** path here so the examples are easy to read and edit.


In [21]:
# Optional: prepend n=2,3,4 (Question, SQL) pairs; compare to Part 3 without few-shot
import re
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

EXAMPLES: list[tuple[str, str]] = [
    ("How many customers are in the database?", "SELECT COUNT(*) AS n FROM Customer"),
    (
        "List the names of three music genres.",
        "SELECT Name FROM Genre LIMIT 3",
    ),
    (
        "For each genre name, how many tracks are there? Show genre and count, limit 5 rows.",
        "SELECT g.Name, COUNT(t.TrackId) AS c FROM Genre g "
        "JOIN Track t ON t.GenreId = g.GenreId GROUP BY g.Name ORDER BY c DESC LIMIT 5",
    ),
    (
        "What are album titles for the artist AC/DC?",
        "SELECT a.Title FROM Album a JOIN Artist ar ON a.ArtistId = ar.ArtistId "
        "WHERE ar.Name = 'AC/DC'",
    ),
]
N: int = 3  # try 2, 3, or 4
shots = EXAMPLES[:N]

def build_few_shot_user(schema_text: str, new_question: str) -> str:
    lines: list[str] = [
        "The following are example pairs (question, then one SQLite SELECT) for the same kind of database.",
    ]
    for i, (q, sql) in enumerate(shots, 1):
        lines.append(f"Example {i} Q: {q}")
        lines.append(f"Example {i} SQL: {sql}")
    lines.append("")
    lines.append("Schema (truncated):")
    lines.append(schema_text)
    lines.append("")
    lines.append("Now output ONLY one SELECT for the new question (no explanation).")
    lines.append("New question: " + new_question)
    return "\n".join(lines)

chat_fs = ChatOllama(model="llama3.2:1b", temperature=0)
schema_short = db.get_table_info()[:3500]
user_q = "What is the email domain used most often among customers? (Hint: use Email and GROUP BY.)"
msg = [
    SystemMessage(content="You output only one SQLite SELECT, same style as the examples."),
    HumanMessage(content=build_few_shot_user(schema_short, user_q)),
]
raw = chat_fs.invoke(msg).content
print("Few-shot (N=" + str(N) + ") model output:\n", raw)
m = re.search(r"(SELECT[\s\S]+?)(;|$)", raw, re.I)
if m:
    sql_out = m.group(1).strip()
    print("--- Executing ---\n", sql_out)
    print(db.run(sql_out))
else:
    print("No SELECT; increase model size or lower N / shorten schema.")


Few-shot (N=3) model output:
 SELECT Email FROM Customer GROUP BY Email ORDER BY COUNT(*) DESC LIMIT 1
--- Executing ---
 SELECT Email FROM Customer GROUP BY Email ORDER BY COUNT(*) DESC LIMIT 1
[('wyatt.girard@yahoo.fr',)]


## Part 4 — Plain-Python **retry** (generate → run → *optional* second prompt)

### Why Part 4 (separate from Part 3)?

- Part 3 (and the optional few-shot block) = **one shot**: one answer from the model, then execute.  
- In practice, **SQLite often errors** (wrong table name, bad join, typo). A **query checker** inside `SQLDatabaseChain` helps a bit, but you may want a **custom repair loop**: log the error, count retries, or change the prompt.  
- **Part 4** shows **generate → `db.run` → on exception, new message with the error** in plain Python, so the pattern is **transparent** and easy to test.

Same staged idea as a larger pipeline, but in one file: (1) ask for SQL, (2) `db.run`, (3) on failure, send the **error** back in a follow-up and try again. Adjust `max_rounds` as you like.

This is **not** a substitute for read-only users, allow-listed tables, and audit logging on real data.


In [22]:
import re
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

# Reuse `db` and path logic from above; re-define helpers so this cell is runnable as a block

def extract_select(text: str) -> str | None:
    t = (text or "").strip()
    if "SELECT" not in t.upper():
        return None
    m = re.search(r"(SELECT[\s\S]+?)(;|$)", t, re.I | re.S)
    if m:
        return m.group(1).strip()
    return None


def ask_chinook_with_retry(
    question: str,
    *,
    model: str = "llama3.2:1b",
    max_rounds: int = 2,
    schema_chars: int = 4000,
) -> tuple[str | None, str]:
    # Returns (last_sql, result_string_from_db_or_error)
    chat = ChatOllama(model=model, temperature=0)
    schema = db.get_table_info()[:schema_chars]
    sys = SystemMessage(
        content="You output ONLY one SQLite SELECT, no code fences, no explanation. Dialect: SQLite."
    )
    last_sql: str | None = None
    for round_i in range(max_rounds):
        if round_i == 0:
            user = f"Schema (truncated):\n{schema}\n\nQuestion: {question}"
        else:
            user = (
                f"Your previous query failed.\nSQL:\n{last_sql}\n\n"
                f"SQLite said:\n{err}\n\nReturn a corrected single SELECT for: {question}"
            )
        raw = chat.invoke([sys, HumanMessage(content=user)]).content
        last_sql = extract_select(raw) or last_sql
        if not last_sql:
            return None, f"No SELECT parsed (round {round_i}): {raw[:500]!r}"
        try:
            return last_sql, db.run(last_sql)
        except Exception as e:  # noqa: BLE001
            err = f"{e!s}"
            if round_i == max_rounds - 1:
                return last_sql, f"SQL error (giving up): {err}"
    return last_sql, "unreachable"


sql, result = ask_chinook_with_retry("How many genres are in the Genre table?")
print("SQL:", sql)
print("Result:", result)


SQL: SELECT COUNT(*) FROM Genre
Result: [(25,)]


##  follow-ups

- Run **2–3** different questions: one aggregate, one **JOIN** across 2+ tables, one *sort/limit* (“top 5 …”).  
- **Compare** `SQLDatabaseChain` vs. manual `ChatOllama` + `db.run` when the model is weak: which path fails first?  
- **(Optional)** Turn **few-shot** `N` from `2` to `4` on the same new question: does SQL quality or stability improve? Report token/latency tradeoff in one line.  
- **Tuning:** try a **larger** Ollama model, shorten `get_table_info()`, or name specific tables in the user question.  
